# Module 3 — Debugging & Failure Modes

> **Time:** 20 minutes.
>
> **What you'll do:** find and fix **4 planted bugs** in your Module 2 system. One bug per failure family from the slides, plus one silent failure.

The system below *was* working at the end of Module 2. Then someone made a few "small changes" and now it misbehaves. Your job: reproduce each issue, classify which failure family it belongs to, and fix it with a **minimal diff**.

**Bug families to expect** (from the slides):

| #   | Family                      | Symptom shape                         |
|-----|-----------------------------|---------------------------------------|
| I   | Hallucination / wrong tool  | Confident but off-policy output       |
| II  | Broken coordination         | Wrong branch, infinite loop, crash    |
| III | State propagation           | Empty / wrong keys, silent corruption |
| —   | Silent failure              | Looks fine but isn't                  |

**Recommended workflow:** run the cells top-to-bottom, observe what breaks, then jump to the matching debugging section below.

> *Solution notebook. Open the starter first — debugging skill comes from doing the search yourself.*

## 1.  Setup (same as Module 2)

In [2]:
%pip install -q \
    langgraph==0.2.* \
    langchain==0.3.* \
    langchain-openai==0.2.* \
    pydantic==2.*


import os

def _ensure_key(name: str, optional: bool = False) -> None:
    """Load an API key from (in order): existing env, Colab Secrets, or getpass.

    Colab Secrets are the recommended path for this workshop — set them ONCE
    via the 🔑 key icon in Colab's left sidebar and every notebook will pick
    them up automatically.
    """
    if os.environ.get(name):
        print(f"  ✓  {name} already set in environment")
        return
    # 1. Try Colab Secrets
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            os.environ[name] = val
            print(f"  ✓  {name} loaded from Colab Secrets")
            return
    except Exception:
        pass
    # 2. Fallback — prompt the user
    import getpass
    val = getpass.getpass(f"Paste your {name}{' (optional)' if optional else ''}: ").strip()
    if val:
        os.environ[name] = val
        print(f"  ✓  {name} set")
    elif optional:
        print(f"  •  {name} skipped (optional)")
    else:
        print(f"  ⚠   {name} skipped — you'll hit errors later without it")

_ensure_key("OPENAI_API_KEY")

print("Ready.")


  ✓  OPENAI_API_KEY loaded from Colab Secrets
Ready.


## 2.  Knowledge base

In [3]:
KB = {
    "billing": [
        {"id": "B-001", "title": "Billing cycle and prorations",
         "text": "We bill on the same day each month. If you change plans mid-cycle, the next invoice is prorated."},
        {"id": "B-002", "title": "Refund policy",
         "text": "Refunds are available within 14 days. Issued to original payment method, settle in 5 business days."},
        {"id": "B-003", "title": "Failed payments",
         "text": "Failed payments retry once a day for 3 days, then 7-day grace period."},
    ],
    "technical": [
        {"id": "T-001", "title": "Login issues",
         "text": "Clear cookies for the domain. For MFA failures, check spam and verify registered phone."},
        {"id": "T-002", "title": "API rate limits",
         "text": "Standard plan: 60 req/min. Enterprise: 600 req/min. Respect Retry-After header."},
        {"id": "T-003", "title": "Export failures",
         "text": "Retry from the same dialog — export jobs are idempotent."},
    ],
    "account": [
        {"id": "A-001", "title": "Password reset",
         "text": "Use Forgot password. Reset email valid for 30 minutes. Check spam."},
        {"id": "A-002", "title": "Account closure",
         "text": "Settings → Account → Close. 30-day pending-deletion window, then permanent deletion."},
        {"id": "A-003", "title": "Ownership transfer",
         "text": "Owner invites new owner as Admin, both confirm via email."},
    ],
}

## 3.  State schema

In [4]:
from typing import TypedDict, Annotated, Literal
from operator import add


class TriageState(TypedDict):
    ticket: str
    category: Literal["billing", "technical", "account"] | None
    urgency:  Literal["low", "med", "high"] | None
    retrieved: list[dict]
    draft: str
    verdict: Literal["pass", "revise"] | None
    revision_count: int
    revisions: Annotated[list[str], add]


def make_initial_state(ticket: str) -> TriageState:
    return {
        "ticket": ticket, "category": None, "urgency": None,
        "retrieved": [], "draft": "", "verdict": None,
        "revision_count": 0, "revisions": [],
    }

## 4.  LLM

In [5]:
from langchain_openai import ChatOpenAI

# FIXED #4 — temperature=0 makes runs reproducible. Module 3 slide 8 covered this.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 5.  Agents and graph — FIXED

In [6]:
import json

def parse_json(text: str) -> dict:
    text = text.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
    return json.loads(text.strip())


CLASSIFY_PROMPT = """\
Categorize the ticket as one of: billing, technical, account.
Rate urgency as: low, med, high.
Return JSON: {{"category": "...", "urgency": "..."}}

TICKET:
{ticket}
"""


# This node is clean — no bug here.
def classify(state: TriageState) -> dict:
    prompt = CLASSIFY_PROMPT.format(ticket=state["ticket"])
    response = llm.invoke(prompt)
    parsed = parse_json(response.content)
    return {"category": parsed["category"], "urgency": parsed["urgency"]}

In [7]:
def retrieve(state: TriageState) -> dict:
    category = state["category"]
    docs = KB.get(category, [])
    # FIXED #3 — the state schema declares 'retrieved', so we write 'retrieved'.
    return {"retrieved": docs}

In [8]:
DRAFT_PROMPT = """\
You are a customer support agent.

Reply to the ticket using ONLY the policies below. If a policy doesn't \
answer the question, say so honestly — do not invent details.

POLICIES:
{context}

TICKET:
{ticket}

Write a concise, helpful reply (3-6 sentences).
"""


def draft(state: TriageState) -> dict:
    # FIXED #1 — the prompt now uses the retrieved context.
    context = "\n\n".join(
        f"[{d['id']}] {d['title']}\n{d['text']}"
        for d in state["retrieved"]
    )
    prompt = DRAFT_PROMPT.format(context=context, ticket=state["ticket"])
    response = llm.invoke(prompt)
    return {
        "draft": response.content,
        "revisions": [response.content],
    }

In [9]:
QA_PROMPT = """\
Decide if the DRAFT is acceptable.
Return JSON: {{"verdict": "pass" | "revise"}}

TICKET:
{ticket}

DRAFT:
{draft}
"""


# This node is clean — no bug here.
def qa(state: TriageState) -> dict:
    prompt = QA_PROMPT.format(ticket=state["ticket"], draft=state["draft"])
    response = llm.invoke(prompt)
    parsed = parse_json(response.content)
    return {
        "verdict": parsed["verdict"],
        "revision_count": state["revision_count"] + 1,
    }

In [10]:
from langgraph.graph import StateGraph, END

MAX_REVISIONS = 2


def route_qa(state: TriageState) -> str:
    # FIXED #2 — return the NODE NAME (key in the mapping), not the raw verdict.
    if state["verdict"] == "pass":
        return "END"
    if state["revision_count"] >= MAX_REVISIONS:
        return "END"
    return "drafter"


graph = StateGraph(TriageState)
graph.add_node("classify", classify)
graph.add_node("retrieve", retrieve)
graph.add_node("drafter",  draft)
graph.add_node("qa",       qa)

graph.set_entry_point("classify")
graph.add_edge("classify", "retrieve")
graph.add_edge("retrieve", "drafter")
graph.add_edge("drafter",  "qa")

graph.add_conditional_edges(
    "qa", route_qa,
    {"drafter": "drafter", "END": END},
)

app = graph.compile()
print("Graph compiled.")

Graph compiled.


/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## 6.  Verify on the three test tickets

In [11]:
verification_tickets = [
    "Hi, my monthly bill is $50 higher than last month. Can you check what happened?",
    "I keep getting 429 errors from your API on the Standard plan.",
    "Forgot my password and the reset email never showed up. Demo in 30 minutes.",
]

for t in verification_tickets:
    try:
        result = app.invoke(make_initial_state(t))
        ok = result["verdict"] is not None and len(result["draft"]) > 50
        print(f"  {'✓' if ok else '✗'}  category={result['category']:9}  "
              f"verdict={result['verdict']:7}  revisions={result['revision_count']}")
    except Exception as e:
        print(f"  ✗  CRASH: {type(e).__name__}: {e}")

  ✓  category=billing    verdict=revise   revisions=2
  ✓  category=technical  verdict=pass     revisions=1
  ✓  category=account    verdict=pass     revisions=2


## 7.  Find and fix the bugs

Below are 4 stub sections — one per bug. For each:

1. **Find** — look at the relevant cell above and identify what's wrong
2. **Classify** — which failure family is this? (`I` / `II` / `III` / `silent`)
3. **Fix** — edit the cell above with a minimal diff, then re-run from there

The numbering below is just a suggested order. You may discover them in a different order — that's fine.

### 🐛  Bug A — Crash on first run

**Symptom:** the graph blows up at the QA → next-node edge. Stack trace mentions a `ValueError` or similar about a mapping key.

**Family?** Fill in below — and patch the cell above that owns this bug.

In [12]:
# Diagnosis after fixing.
DIAGNOSIS = {
    "bug": "A",
    "family": "II",   # broken coordination
    "what_was_wrong": (
        "route_qa returned the verdict ('pass' / 'revise') directly. "
        "add_conditional_edges expects a NODE NAME — one of the keys in the mapping."
    ),
    "what_i_fixed": (
        "Map verdict to a node name: 'pass' → 'END' (with the revision_count guard), "
        "'revise' → 'drafter'."
    ),
}
print(DIAGNOSIS)

{'bug': 'A', 'family': 'II', 'what_was_wrong': "route_qa returned the verdict ('pass' / 'revise') directly. add_conditional_edges expects a NODE NAME — one of the keys in the mapping.", 'what_i_fixed': "Map verdict to a node name: 'pass' → 'END' (with the revision_count guard), 'revise' → 'drafter'."}


### 🐛  Bug B — Drafts look generic / hallucinated

**Symptom:** after fixing the crash, the draft replies look plausible but don't actually reference the policies in our KB. Run a billing ticket — the draft should mention prorations, but doesn't.

**Family?** Two possible places to look. Check Retriever AND Drafter.

In [13]:
DIAGNOSIS = {
    "bug": "B",
    "family": "III",   # state propagation
    "what_was_wrong": (
        "Retriever wrote to state['docs'], but TriageState declares 'retrieved' "
        "and Drafter reads state['retrieved']. The handoff was silently dropping data."
    ),
    "what_i_fixed": (
        "Change retrieve() to return {'retrieved': docs}."
    ),
}
print(DIAGNOSIS)

{'bug': 'B', 'family': 'III', 'what_was_wrong': "Retriever wrote to state['docs'], but TriageState declares 'retrieved' and Drafter reads state['retrieved']. The handoff was silently dropping data.", 'what_i_fixed': "Change retrieve() to return {'retrieved': docs}."}


### 🐛  Bug C — Drafts *still* don't use the policies

**Symptom:** even after fixing the state-key mismatch, the drafts still don't ground in the retrieved policies. The data is in state, but the LLM doesn't see it.

**Family?** Look at the prompt template, not just the function body.

In [14]:
DIAGNOSIS = {
    "bug": "C",
    "family": "I",   # hallucination — wrong tool / context use
    "what_was_wrong": (
        "DRAFT_PROMPT never references {context}. The Retriever's data was correctly "
        "in state, but the prompt never told the LLM about it."
    ),
    "what_i_fixed": (
        "Added a POLICIES section to DRAFT_PROMPT, formatted with {context}, and "
        "added 'use ONLY the policies below' to the instructions."
    ),
}
print(DIAGNOSIS)

{'bug': 'C', 'family': 'I', 'what_was_wrong': "DRAFT_PROMPT never references {context}. The Retriever's data was correctly in state, but the prompt never told the LLM about it.", 'what_i_fixed': "Added a POLICIES section to DRAFT_PROMPT, formatted with {context}, and added 'use ONLY the policies below' to the instructions."}


### 🐛  Bug D — Different runs give different drafts

**Symptom:** rerun the same ticket twice. The drafts come back materially different. Production reproducibility is shot — eval (Module 5) will be useless.

**Family?** Silent failure. The graph works; the *behavior* doesn't.

In [15]:
DIAGNOSIS = {
    "bug": "D",
    "family": "silent",
    "what_was_wrong": (
        "ChatOpenAI was initialized with temperature=0.7. Same input → different output. "
        "No crash, no wrong key, no hallucination — just unreproducible runs."
    ),
    "what_i_fixed": (
        "Set temperature=0 in the ChatOpenAI() call. Module 3 slide 8: 'pin determinism'."
    ),
}
print(DIAGNOSIS)

{'bug': 'D', 'family': 'silent', 'what_was_wrong': 'ChatOpenAI was initialized with temperature=0.7. Same input → different output. No crash, no wrong key, no hallucination — just unreproducible runs.', 'what_i_fixed': "Set temperature=0 in the ChatOpenAI() call. Module 3 slide 8: 'pin determinism'."}


## Wrap up

You used Module 3's debugging loop end-to-end:

1. **Reproduced** — let the system actually fail
2. **Isolated** — used `stream()` to see which node's output went wrong
3. **Hypothesized** — matched the symptom to a failure family
4. **Fixed** — minimal diff per bug
5. **Verified** — reran the whole suite to catch regressions

**Up next:** Module 4 — Observability. Instead of `stream()` and print, we wire up LangSmith — and the same investigations get a lot faster.